# Cuaderno U1-05. Alcances y limitaciones de los modelos

**Modelacion y Simulacion Computacional** · Maestria en Ingenieria · Universidad de Sucre

Unidad 1, Fundamentos de modelacion en ingenieria · Subtema 1.5 del plan de asignatura

Docente Daniel David Otero Meza · Periodo 2026-2

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad1/U1_05_alcances_y_limitaciones.ipynb)


Este cuaderno acompana la Seccion 1.6 del libro y recoge de la Seccion 1.5 los dos episodios que cuantifican hasta donde llega una simplificacion. Delimita el dominio de validez de una correlacion empirica, mide el costo de la hipotesis de resistencia interna despreciable, comprueba cuando deja de valer la superposicion, descompone el error de una prediccion y cierra con el protocolo de lectura critica del modelo ajeno.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante estara en capacidad de hacer lo siguiente.

1. Delimitar el dominio de validez de un modelo empirico y cuantificar el error que se comete al extrapolarlo, como hace el Ejemplo 1.6.
2. Medir el costo en exactitud de una simplificacion justificada por un grupo adimensional, como hace el Ejemplo 1.4 con el numero de Biot.
3. Comprobar la linealidad de un problema antes de superponer soluciones elementales, como exige el Ejemplo 1.5.
4. Descomponer el error de una prediccion en sus cuatro componentes y elegir la accion que efectivamente reduce cada uno.
5. Propagar la incertidumbre de un parametro por primer orden y contrastarla con Monte Carlo.
6. Aplicar el protocolo de lectura critica a un modelo ajeno.

## Puesta a punto

La primera celda detecta el entorno e instala unicamente lo que falte. La segunda fija la semilla del curso y la paleta del libro. La tercera define las funciones de verificacion que se usan mas abajo. Ejecutelas en orden antes de continuar.

In [ ]:
# Puesta a punto del entorno. Detecta Colab e instala solo lo que falte.
import importlib
import importlib.util
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes):
    """Instala los paquetes ausentes sin reinstalar los que ya estan."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)
    return faltantes


AUSENTES = asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
                     "matplotlib": "matplotlib", "sympy": "sympy"})

print("Entorno de ejecucion:", "Google Colab" if EN_COLAB else "JupyterLab local")
print("Paquetes instalados en esta sesion:", AUSENTES or "ninguno, ya estaban")

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

# Semilla unica de la asignatura. Ningun resultado depende de una corrida.
SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

# Paleta del libro. Los cuadernos usan los mismos colores que las figuras.
PALETA = {
    "azul": "#1F4E79",
    "rojo": "#B3251E",
    "verde": "#2E7D32",
    "naranja": "#E07B00",
    "gris": "#5A5A5A",
    "morado": "#6A3D9A",
}

plt.rcParams.update({
    "figure.figsize": (8.6, 4.6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
    "font.size": 10.0,
    "legend.frameon": True,
    "legend.framealpha": 0.92,
})

print(f"NumPy {np.__version__} · SciPy {scipy.__version__} · pandas {pd.__version__}")
print(f"SymPy {sp.__version__} · semilla del curso {SEMILLA}")

In [ ]:
# Bandera de revision de los ejercicios guiados.
# Mientras valga False el cuaderno se ejecuta completo aunque falten celdas.
# Pongala en True cuando haya completado las celdas marcadas para completar.
REVISAR = False
print("REVISAR =", REVISAR)

In [ ]:
def verificar_libro(nombre, obtenido, publicado, tolerancia=1e-3, unidad=""):
    """Contrasta un resultado calculado con la cifra que publica el libro."""
    valor = float(obtenido)
    escala = abs(publicado) if publicado else 1.0
    error = abs(valor - publicado) / escala
    print(f"{nombre:<46s} calculado {valor:>12.6g} {unidad:<10s}"
          f" libro {publicado:>12.6g}  error rel. {error:.1e}")
    assert error <= tolerancia, f"{nombre} se aparta de la cifra publicada"
    return valor


def comprobar(nombre, obtenido, referencia, tolerancia=1e-3, unidad=""):
    """Revisa una celda de ejercicio contra su valor de referencia.

    Con REVISAR en False solo informa que el ejercicio sigue pendiente, de modo
    que el cuaderno nunca se detiene por una celda sin completar.
    """
    if not REVISAR:
        print(f"[pendiente]  {nombre}")
        return False
    valor = float(obtenido)
    escala = abs(referencia) if referencia else 1.0
    error = abs(valor - referencia) / escala
    marca = "correcto " if error <= tolerancia else "revisar  "
    print(f"[{marca}]  {nombre} = {valor:.6g} {unidad}"
          f"  referencia {referencia:.6g}  error rel. {error:.2e}")
    assert error <= tolerancia, f"{nombre} no coincide con la referencia"
    return True


def ruta_datos(nombre):
    """Ubica un archivo de la carpeta datos sin usar rutas absolutas.

    Funciona igual en Colab, donde el cuaderno suele abrirse en el directorio
    de trabajo, y en una copia local del repositorio, donde el cuaderno vive
    dentro de Unidad1 o de soluciones.
    """
    candidatas = (Path("datos"),
                  Path("..") / "datos",
                  Path("..") / ".." / "datos",
                  Path("03_cuadernos") / "datos")
    for base in candidatas:
        if (base / nombre).exists():
            return base / nombre
    for base in candidatas:        # la carpeta existe pero el archivo aun no
        if base.is_dir():
            return base / nombre
    return Path("datos") / nombre  # entorno nuevo, como una sesion de Colab


def cargar_o_generar(nombre, generador):
    """Lee el archivo de datos y, si no esta, lo reconstruye con la semilla."""
    ruta = ruta_datos(nombre)
    if ruta.exists():
        print(f"Datos leidos de {ruta}")
        return pd.read_csv(ruta)
    tabla = generador()
    ruta.parent.mkdir(parents=True, exist_ok=True)
    tabla.to_csv(ruta, index=False)
    print(f"Datos regenerados con la semilla {SEMILLA} y guardados en {ruta}")
    return tabla


def integrar_trapecio(valores, muestras):
    """Regla del trapecio compatible con NumPy 1 y con NumPy 2."""
    regla = getattr(np, "trapezoid", None) or np.trapz
    return float(regla(valores, muestras))


print("Funciones auxiliares disponibles.")

## 1. El dominio de validez

La Definicion 1.13 del libro llama dominio de validez al conjunto de valores de
las entradas, de los parametros y de las condiciones de operacion para los
cuales se ha comprobado, contra evidencia independiente, que el error de la
respuesta no supera la tolerancia declarada. Fuera de ese conjunto el modelo
sigue produciendo numeros, pero ya no produce predicciones respaldadas.

Omitir esa delimitacion es la forma mas frecuente de mala practica profesional,
porque un numero presentado sin su dominio de validez invita a usarlo donde no
vale.

### Ejemplo 1.6 del libro, extrapolar una correlacion fotovoltaica

Una campana de medicion sobre un modulo fotovoltaico registra la potencia
entregada frente a la irradiancia en el plano del modulo, en diez puntos
repartidos entre 700 y 1000 W/m2, que son las condiciones del mediodia despejado
en que se hizo la campana. Se ajusta una recta y se pretende usarla para estimar
la energia diaria del modulo.

Los datos se leen de la carpeta `datos` y, si no estuvieran disponibles, se
reconstruyen con la semilla del curso a partir del comportamiento de referencia
que declara el enunciado del ejemplo, el cual incorpora la caida de rendimiento
a baja irradiancia y el efecto de la temperatura de celda.

In [ ]:
POTENCIA_STC = 330.0      # W en condiciones estandar
COEF_TEMPERATURA = -3.8e-3   # 1/K, esto es -0.38 por ciento por kelvin
COEF_BAJA_IRRADIANCIA = 0.045  # adimensional
NOCT = 45.0               # C, temperatura nominal de operacion de la celda
TEMPERATURA_AIRE = 30.0   # C


def temperatura_celda(irradiancia):
    """Temperatura de celda por el modelo NOCT, en C."""
    return TEMPERATURA_AIRE + (NOCT - 20.0) / 800.0 * np.asarray(irradiancia, dtype=float)


def potencia_referencia(irradiancia):
    """Comportamiento de referencia del modulo, en W."""
    g = np.maximum(np.asarray(irradiancia, dtype=float), 1e-9)
    factor_termico = 1.0 + COEF_TEMPERATURA * (temperatura_celda(g) - 25.0)
    factor_irradiancia = 1.0 + COEF_BAJA_IRRADIANCIA * np.log(g / 1000.0)
    return (g / 1000.0) * POTENCIA_STC * factor_termico * factor_irradiancia


def generar_campana_fotovoltaica():
    """Los diez puntos de la campana, con la semilla del curso."""
    generador = np.random.default_rng(SEMILLA)
    irradiancia = np.linspace(700.0, 1000.0, 10)
    potencia = potencia_referencia(irradiancia) + generador.normal(0.0, 1.2, 10)
    return pd.DataFrame({"irradiancia_W_m2": np.round(irradiancia, 4),
                         "potencia_W": np.round(potencia, 6)})


campana = cargar_o_generar("U1_modulo_fotovoltaico.csv", generar_campana_fotovoltaica)
g_medida = campana["irradiancia_W_m2"].to_numpy(dtype=float)
p_medida = campana["potencia_W"].to_numpy(dtype=float)
campana

In [ ]:
def ajustar_recta(irradiancia, potencia):
    """Pendiente, corte y coeficiente de determinacion del ajuste lineal."""
    pendiente, corte = np.polyfit(irradiancia, potencia, 1)
    residuo = potencia - (pendiente * irradiancia + corte)
    varianza = float(((potencia - potencia.mean())**2).sum())
    return pendiente, corte, 1.0 - float(residuo @ residuo) / varianza


pendiente, corte, r_cuadrado = ajustar_recta(g_medida, p_medida)
recta = lambda g: pendiente * np.asarray(g, dtype=float) + corte
irradiancia_nula = -corte / pendiente

verificar_libro("pendiente de la recta", pendiente, 0.2644, 1e-3, "W m2/W")
verificar_libro("corte de la recta", corte, 20.32, 1e-3, "W")
verificar_libro("coeficiente de determinacion", r_cuadrado, 0.9979, 1e-3)
verificar_libro("irradiancia de potencia nula", irradiancia_nula, -76.9, 1e-3, "W/m2")
print(f"\nP = {pendiente:.4f} G + {corte:.2f} W con R2 = {r_cuadrado:.4f}")
print("El termino independiente positivo afirma que el modulo entrega "
      f"{corte:.1f} W con irradiancia nula, lo cual es fisicamente imposible.")

### El error de extrapolar

El libro reporta que a 100 W/m2 la recta sobrestima la potencia en un 63.1 por
ciento y que el error se mantiene por encima del 5 por ciento para toda
irradiancia inferior a 447 W/m2. Al integrar el dia completo, con el perfil de
semiseno entre las seis y las dieciocho horas, la energia estimada supera la de
referencia en un 2.9 por ciento en jornada despejada y en un 21.0 por ciento en
jornada nublada.

In [ ]:
from scipy.optimize import brentq


def error_relativo(irradiancia):
    """Error relativo de la correlacion frente a la referencia, adimensional."""
    return (recta(irradiancia) - potencia_referencia(irradiancia)) / potencia_referencia(irradiancia)


def energia_diaria(potencia, irradiancia_maxima=950.0, puntos=20001):
    """Energia de un dia con perfil de semiseno entre las 6 y las 18 h, en Wh."""
    horas = np.linspace(6.0, 18.0, puntos)
    irradiancia = irradiancia_maxima * np.sin(np.pi * (horas - 6.0) / 12.0)
    return integrar_trapecio(np.maximum(potencia(irradiancia), 0.0), horas)


error_100 = 100.0 * error_relativo(100.0)
umbral_cinco = brentq(lambda g: abs(error_relativo(g)) - 0.05, 120.0, 699.0)

verificar_libro("error a 100 W/m2", error_100, 63.1, 1e-3, "%")
verificar_libro("irradiancia donde el error baja del 5 %", umbral_cinco, 447, 5e-3, "W/m2")

for maxima, jornada, publicado in ((950.0, "despejada", 2.9), (350.0, "nublada", 21.0)):
    real = energia_diaria(potencia_referencia, maxima)
    lineal = energia_diaria(recta, maxima)
    verificar_libro(f"exceso de energia en jornada {jornada}",
                    100.0 * (lineal - real) / real, publicado, 2e-2, "%")

In [ ]:
malla_g = np.linspace(60.0, 1150.0, 800)
dentro = (malla_g >= 700.0) & (malla_g <= 1000.0)

figura, (eje1, eje2) = plt.subplots(2, 1, figsize=(8.6, 6.6), sharex=True,
                                    gridspec_kw={"height_ratios": [1.55, 1.0],
                                                 "hspace": 0.12})
for eje in (eje1, eje2):
    eje.axvspan(700.0, 1000.0, color=PALETA["azul"], alpha=0.10, lw=0)

eje1.plot(malla_g, potencia_referencia(malla_g), color=PALETA["verde"], lw=1.9,
          label="Comportamiento del modulo")
eje1.plot(malla_g[~dentro], recta(malla_g[~dentro]), color=PALETA["rojo"],
          lw=1.5, ls="--", label="Correlacion lineal extrapolada")
eje1.plot(malla_g[dentro], recta(malla_g[dentro]), color=PALETA["rojo"], lw=2.1,
          label="Correlacion lineal ajustada")
eje1.plot(g_medida, p_medida, "o", color=PALETA["azul"], ms=4.6,
          label="Campana de medicion")
eje1.set_ylabel("Potencia entregada (W)")
eje1.set_ylim(0.0, 380.0)
eje1.legend(loc="upper left", fontsize=8.6)
eje1.set_title("Ejemplo 1.6 del libro, Figura 1.12")

error_malla = 100.0 * error_relativo(malla_g)
eje2.axhline(0.0, color=PALETA["gris"], lw=0.8)
eje2.fill_between(malla_g, -5.0, 5.0, color=PALETA["verde"], alpha=0.12, lw=0)
eje2.plot(malla_g[~dentro], error_malla[~dentro], color=PALETA["rojo"], lw=1.5, ls="--")
eje2.plot(malla_g[dentro], error_malla[dentro], color=PALETA["rojo"], lw=2.1)
eje2.set_xlabel("Irradiancia en el plano del modulo (W/m2)")
eje2.set_ylabel("Error relativo (%)")
eje2.set_xlim(60.0, 1150.0)
eje2.set_ylim(-12.0, 72.0)
eje2.annotate("banda de mas o menos 5 %", (830.0, 5.0), xytext=(600.0, 20.0),
              color=PALETA["verde"], fontsize=8.6,
              arrowprops=dict(arrowstyle="-|>", color=PALETA["verde"], lw=0.7))
eje2.annotate("dominio de validez", (850.0, -8.4), ha="center", va="center",
              color=PALETA["azul"], fontsize=8.6)
plt.show()

print("Un coeficiente de determinacion alto no certifica un modelo, porque solo "
      "informa sobre el ajuste dentro de la ventana medida.")
print("Un dimensionamiento de baterias basado en la energia de dias nublados, "
      "que es el caso critico, quedaria con un deficit cercano al veinte por ciento.")

## 2. El costo de una simplificacion justificada

La Definicion 1.12 del libro llama adimensionalizacion al procedimiento que
refiere cada variable a una escala caracteristica del propio problema, de modo
que despreciar un termino deja de ser una licencia y se convierte en una
desigualdad verificable. La Tabla 1.4 reune los grupos de uso frecuente con su
umbral de decision, entre ellos el numero de Biot, que declara isotermico a un
solido cuando es menor que 0.1.

El Ejemplo 1.4 del libro pone precio a esa declaracion. Una lamina de pulpa de
fruta concentrada de 12 mm de espesor se enfria por sus dos caras en una camara
con aire a 20 grados, desde 80 grados, con conductividad 0.52 W/(m K), densidad
1080 kg/m3, calor especifico 3600 J/(kg K) y coeficiente convectivo
8.0 W/(m2 K). Se pide el tiempo necesario para descender hasta 32 grados y el
error que introduce suponer resistencia interna despreciable.

In [ ]:
CONDUCTIVIDAD = 0.52      # W/(m K)
DENSIDAD_PULPA = 1080.0   # kg/m3
CALOR_ESPECIFICO = 3600.0  # J/(kg K)
CONVECCION = 8.0          # W/(m2 K)
SEMIESPESOR = 0.006       # m
TEMPERATURA_INICIAL = 80.0    # C
TEMPERATURA_AMBIENTE = 20.0   # C
TEMPERATURA_OBJETIVO = 32.0   # C


def theta_centro(biot, fourier, terminos=40):
    """Temperatura reducida del plano central por la serie de la Ecuacion 1.8."""
    raices = np.array([
        brentq(lambda x: x * np.tan(x) - biot,
               m * np.pi + 1e-13, (m + 0.5) * np.pi - 1e-13)
        for m in range(terminos)])
    coeficientes = 4 * np.sin(raices) / (2 * raices + np.sin(2 * raices))
    return float(np.sum(coeficientes * np.exp(-raices**2 * fourier)))


def tiempos_enfriamiento(h, k, rho, cp, semiespesor, objetivo):
    """Numero de Biot y tiempos isotermico y exacto, en s."""
    biot = h * semiespesor / k
    difusividad = k / (rho * cp)
    isotermico = -(rho * cp * semiespesor / h) * np.log(objetivo)
    fourier = brentq(lambda f: theta_centro(biot, f) - objetivo, 1e-6, 500.0)
    return biot, isotermico, fourier, fourier * semiespesor**2 / difusividad


theta_objetivo = ((TEMPERATURA_OBJETIVO - TEMPERATURA_AMBIENTE)
                  / (TEMPERATURA_INICIAL - TEMPERATURA_AMBIENTE))
biot, t_isotermico, fourier_exacto, t_exacto = tiempos_enfriamiento(
    CONVECCION, CONDUCTIVIDAD, DENSIDAD_PULPA, CALOR_ESPECIFICO,
    SEMIESPESOR, theta_objetivo)
difusividad = CONDUCTIVIDAD / (DENSIDAD_PULPA * CALOR_ESPECIFICO)
constante_tiempo = DENSIDAD_PULPA * CALOR_ESPECIFICO * SEMIESPESOR / CONVECCION
primera_raiz = brentq(lambda x: x * np.tan(x) - biot, 1e-13, np.pi / 2 - 1e-13)
primer_coeficiente = (4 * np.sin(primera_raiz)
                      / (2 * primera_raiz + np.sin(2 * primera_raiz)))

verificar_libro("numero de Biot", biot, 0.0923, 1e-3)
verificar_libro("difusividad termica", difusividad, 1.337e-7, 1e-3, "m2/s")
verificar_libro("temperatura reducida objetivo", theta_objetivo, 0.20, 1e-9)
verificar_libro("constante de tiempo isotermica", constante_tiempo, 2916, 1e-3, "s")
verificar_libro("tiempo del modelo isotermico", t_isotermico, 4693, 1e-3, "s")
verificar_libro("tiempo isotermico en minutos", t_isotermico / 60, 78.2, 1e-3, "min")
verificar_libro("primera raiz de la ecuacion caracteristica", primera_raiz, 0.2992, 1e-3)
verificar_libro("primer coeficiente de la serie", primer_coeficiente, 1.0149, 1e-3)
verificar_libro("numero de Fourier exacto", fourier_exacto, 18.14, 1e-3)
verificar_libro("tiempo de la solucion exacta", t_exacto, 4883, 1e-3, "s")
verificar_libro("tiempo exacto en minutos", t_exacto / 60, 81.4, 1e-3, "min")

In [ ]:
diferencia = t_exacto - t_isotermico
temperatura_al_predicho = TEMPERATURA_AMBIENTE + 60.0 * theta_centro(
    biot, t_isotermico * difusividad / SEMIESPESOR**2)

verificar_libro("subestimacion del tiempo", diferencia, 190, 2e-2, "s")
verificar_libro("subestimacion relativa", 100 * diferencia / t_exacto, 3.9, 5e-3, "%")
verificar_libro("temperatura del centro al tiempo isotermico",
                temperatura_al_predicho, 32.8, 5e-3, "C")

# Barrido del numero de Biot, que reproduce la Figura 1.11 del libro.
biots = np.geomspace(5e-3, 2.0, 24)
subestimacion = []
for valor in biots:
    fourier_concentrado = -np.log(0.20) / valor
    fourier_exacto_i = brentq(lambda f: theta_centro(valor, f) - 0.20, 1e-6, 5000.0)
    subestimacion.append(100.0 * (fourier_exacto_i - fourier_concentrado) / fourier_exacto_i)
subestimacion = np.array(subestimacion)

figura, eje = plt.subplots()
eje.axvspan(5e-3, 0.1, color=PALETA["verde"], alpha=0.10, lw=0)
eje.plot(biots, subestimacion, color=PALETA["rojo"], lw=1.9,
         label="Temperatura del plano central")
eje.axvline(0.1, color=PALETA["verde"], lw=1.2)
eje.axhline(5.0, color=PALETA["gris"], lw=0.9, ls=(0, (4, 3)))
eje.plot([biot], [100 * diferencia / t_exacto], "o", color=PALETA["azul"], ms=7,
         zorder=5, label="Lamina de pulpa del Ejemplo 1.4")
eje.set_xscale("log")
eje.set_yscale("log")
eje.set_xlim(5e-3, 2.0)
eje.set_ylim(0.1, 60.0)
eje.set_xlabel("Numero de Biot (adimensional)")
eje.set_ylabel("Subestimacion del tiempo de enfriamiento (%)")
eje.legend(loc="upper left", fontsize=8.8)
eje.set_title("Ejemplo 1.4 del libro, costo de la hipotesis de solido isotermico")
plt.show()

en_umbral = float(np.interp(0.1, biots, subestimacion))
en_medio = float(np.interp(0.5, biots, subestimacion))
print(f"En el umbral Bi = 0.1 la subestimacion vale {en_umbral:.1f} por ciento "
      f"y en Bi = 0.5 sube a {en_medio:.0f} por ciento.")
print("La simplificacion esta justificada aqui porque ese error es despreciable "
      "frente a la variabilidad del espesor de lamina en una linea real. La "
      "conclusion cambiaria por completo con enfriamiento por inmersion en agua.")

## 3. El limite de la linealidad

El Teorema 1.3 del libro autoriza construir soluciones complejas sumando
soluciones elementales, siempre que el operador sea lineal. Basta un termino no
lineal para invalidarlo.

El Ejemplo 1.5 lo ilustra en un acuifero confinado de transmisividad
250 m2/dia y coeficiente de almacenamiento 1.5e-4, explotado con dos pozos que
bombean de forma continua 900 y 1200 m3/dia. Se pide el descenso al cabo de
30 dias en un punto situado a 144.2 m del primer pozo y a 197.0 m del segundo.
El libro reporta 2.463 m del primer pozo, 3.046 m del segundo y 5.509 m en
total.

In [ ]:
from scipy.special import exp1

TRANSMISIVIDAD = 250.0     # m2/dia
ALMACENAMIENTO = 1.5e-4    # adimensional
TIEMPO_BOMBEO = 30.0       # dias


def cooper_jacob(caudal, radio, transmisividad=TRANSMISIVIDAD,
                 almacenamiento=ALMACENAMIENTO, tiempo=TIEMPO_BOMBEO):
    """Descenso por la aproximacion logaritmica y su argumento u."""
    u = radio**2 * almacenamiento / (4 * transmisividad * tiempo)
    descenso = (2.303 * caudal / (4 * np.pi * transmisividad)
                * np.log10(2.25 * transmisividad * tiempo
                           / (radio**2 * almacenamiento)))
    return descenso, u


def theis(caudal, radio, transmisividad=TRANSMISIVIDAD,
          almacenamiento=ALMACENAMIENTO, tiempo=TIEMPO_BOMBEO):
    """Descenso por la solucion exacta de Theis, en m."""
    u = radio**2 * almacenamiento / (4 * transmisividad * tiempo)
    return caudal / (4 * np.pi * transmisividad) * float(exp1(u))


POZOS = ((900.0, 144.2), (1200.0, 197.0))
descensos = [cooper_jacob(q, r) for q, r in POZOS]
descenso_total = sum(d for d, _ in descensos)
exactos = [theis(q, r) for q, r in POZOS]

verificar_libro("descenso del primer pozo", descensos[0][0], 2.463, 1e-3, "m")
verificar_libro("descenso del segundo pozo", descensos[1][0], 3.046, 1e-3, "m")
verificar_libro("descenso total por superposicion", descenso_total, 5.509, 1e-3, "m")
verificar_libro("argumento u del primer pozo", descensos[0][1], 1.04e-4, 5e-3)
verificar_libro("argumento u del segundo pozo", descensos[1][1], 1.94e-4, 5e-3)
verificar_libro("descenso exacto del primer pozo", exactos[0], 2.462, 1e-3, "m")
verificar_libro("descenso exacto del segundo pozo", exactos[1], 3.045, 1e-3, "m")

diferencia_relativa = 100 * abs(descensos[0][0] - exactos[0]) / exactos[0]
print(f"\nAmbos argumentos son muy inferiores a 0.01, de modo que la "
      "aproximacion logaritmica es aplicable.")
print(f"La diferencia relativa frente a Theis vale {diferencia_relativa:.2f} "
      "por ciento.")

In [ ]:
# Donde deja de valer la superposicion. En un acuifero libre el espesor saturado
# disminuye al descender el nivel y la ecuacion se vuelve no lineal, de modo que
# la suma de descensos individuales subestima el conjunto.
ESPESOR_SATURADO = 22.0   # m


def descenso_libre(descenso_confinado, espesor=ESPESOR_SATURADO):
    """Correccion de Jacob para acuifero libre, en m."""
    return espesor * (1.0 - np.sqrt(np.maximum(1.0 - 2.0 * descenso_confinado / espesor, 0.0)))


suma_individual = descenso_libre(descensos[0][0]) + descenso_libre(descensos[1][0])
conjunto = descenso_libre(descenso_total)

print(f"acuifero confinado, superposicion exacta   · {descenso_total:.3f} m")
print(f"acuifero libre, suma de descensos aislados · {suma_individual:.3f} m")
print(f"acuifero libre, descenso conjunto correcto · {conjunto:.3f} m")
print(f"la suma subestima el descenso real en "
      f"{100 * (conjunto - suma_individual) / conjunto:.1f} por ciento")
assert conjunto > suma_individual
print("\nComprobar la linealidad antes de superponer es tan importante como "
      "saber superponer.")

## 4. Las cuatro componentes del error

El error de una prediccion no proviene de una sola causa, y por eso perfeccionar
un solo ingrediente rara vez mejora el resultado. La Tabla 1.5 del libro lo
descompone en cuatro contribuciones e indica como se detecta y como se reduce
cada una. Reducir una sola componente por debajo de las demas no mejora la
prediccion, solo desplaza el costo.

In [ ]:
FUENTES = [
    {"componente": "estructural", "origen": "procesos omitidos o mal representados",
     "senal caracteristica": "residuos con patron sistematico",
     "accion que lo reduce": "revisar el modelo conceptual"},
    {"componente": "de parametros", "origen": "estimacion con datos limitados",
     "senal caracteristica": "intervalos de confianza amplios o correlacionados",
     "accion que lo reduce": "mas datos informativos o menos parametros"},
    {"componente": "de datos de entrada", "origen": "instrumentacion y series de forzamiento",
     "senal caracteristica": "dispersion que no cambia con el modelo",
     "accion que lo reduce": "mejor medicion y control de calidad"},
    {"componente": "numerico", "origen": "discretizacion y aritmetica finita",
     "senal caracteristica": "el resultado cambia al refinar",
     "accion que lo reduce": "refinar hasta que el cambio sea despreciable"},
]
pd.DataFrame(FUENTES).set_index("componente")

### La componente de parametros, por primer orden y por Monte Carlo

El Teorema 1.4 del libro dice que, en la aproximacion de primer orden, la
varianza de la salida es la forma cuadratica de la covarianza de las entradas
evaluada en el gradiente. Aplicado al canal del Ejemplo 1.2, con rugosidad de
valor esperado 0.025 y desviacion estandar 0.0025, el libro reporta una derivada
del tirante respecto de la rugosidad de 36.6 m por unidad de coeficiente y una
desviacion estandar de 0.091 m. Una simulacion con cinco mil realizaciones y la
semilla del curso entrega 0.092 m y un intervalo central del noventa y cinco por
ciento entre 1.700 m y 2.056 m.

In [ ]:
GRAVEDAD = 9.81


def tirante_normal(caudal, ancho, talud, rugosidad, pendiente):
    """Tirante de flujo uniforme del canal trapezoidal, en m."""
    area = lambda y: (ancho + talud * y) * y
    perimetro = lambda y: ancho + 2.0 * y * np.sqrt(1.0 + talud**2)

    def residuo(y):
        radio = area(y) / perimetro(y)
        return area(y) * radio**(2 / 3) * np.sqrt(pendiente) / rugosidad - caudal

    return brentq(residuo, 1e-4, 10.0, xtol=1e-12)


modelo_canal = lambda n: tirante_normal(12.0, 2.50, 1.5, n, 0.0008)


def propagacion_primer_orden(modelo, valor_medio, desviacion, paso=1e-6):
    """Valor central y desviacion estandar de la salida, por primer orden."""
    derivada = (modelo(valor_medio + paso) - modelo(valor_medio - paso)) / (2 * paso)
    return modelo(valor_medio), abs(derivada) * desviacion, derivada


def monte_carlo(modelo, valor_medio, desviacion, muestras=5000, semilla=SEMILLA):
    """Media, desviacion e intervalo central del 95 por ciento."""
    generador = np.random.default_rng(semilla)
    respuesta = np.array([modelo(v) for v in
                          generador.normal(valor_medio, desviacion, muestras)])
    return (float(respuesta.mean()), float(respuesta.std(ddof=1)),
            np.percentile(respuesta, [2.5, 97.5]), respuesta)


central, sigma_primer_orden, derivada = propagacion_primer_orden(modelo_canal, 0.025, 0.0025)
media_mc, sigma_mc, intervalo, realizaciones = monte_carlo(modelo_canal, 0.025, 0.0025)

verificar_libro("derivada del tirante respecto de n", derivada, 36.6, 1e-2, "m")
verificar_libro("desviacion por primer orden", sigma_primer_orden, 0.091, 1e-2, "m")
verificar_libro("desviacion por Monte Carlo", sigma_mc, 0.092, 1e-2, "m")
verificar_libro("percentil 2.5 del tirante", float(intervalo[0]), 1.700, 1e-2, "m")
verificar_libro("percentil 97.5 del tirante", float(intervalo[1]), 2.056, 1e-2, "m")

figura, eje = plt.subplots()
eje.hist(realizaciones, bins=45, color=PALETA["azul"], alpha=0.72,
         edgecolor="white", linewidth=0.4, label="5000 realizaciones")
eje.axvline(central, color=PALETA["rojo"], lw=1.8,
            label=f"Tirante de diseno, {central:.3f} m")
eje.axvline(float(intervalo[0]), color=PALETA["naranja"], lw=1.3, ls="--",
            label=f"Intervalo central del 95 %, [{intervalo[0]:.3f}, {intervalo[1]:.3f}] m")
eje.axvline(float(intervalo[1]), color=PALETA["naranja"], lw=1.3, ls="--")
eje.set_xlabel("Tirante normal (m)")
eje.set_ylabel("Frecuencia (numero de realizaciones)")
eje.legend(loc="upper right", fontsize=8.6)
eje.set_title("Propagacion de la incertidumbre del coeficiente de rugosidad")
plt.show()

asimetria = ((float(intervalo[1]) - media_mc) - (media_mc - float(intervalo[0])))
print(f"asimetria del intervalo respecto de la media · {asimetria:+.4f} m")
print("El intervalo es levemente asimetrico porque el tirante crece mas rapido "
      "que linealmente con la rugosidad, de modo que la formula de primer orden "
      "subestima el riesgo del lado desfavorable.")

### Parsimonia y equifinalidad

Anadir procesos reduce el error estructural y aumenta el numero de parametros
por estimar, de suerte que a partir de cierto punto el error de estimacion crece
mas rapido de lo que el estructural disminuye. La Figura 1.13 del libro muestra
ese valle y su desplazamiento hacia mayor complejidad cuando se dispone de mas
datos.

Con esa penalizacion aparece la equifinalidad, que consiste en que conjuntos de
valores muy distintos reproducen las observaciones con calidad casi identica.
No es un defecto del algoritmo sino una propiedad del problema, ligada a que los
datos no contienen informacion suficiente para separar los efectos de los
parametros. La celda siguiente la exhibe sobre una suma de dos exponenciales
ajustada a datos con ruido moderado.

In [ ]:
from scipy.optimize import minimize

tiempos_obs = np.linspace(0.2, 6.0, 14)
observaciones = (0.6 * np.exp(-0.45 * tiempos_obs)
                 + 0.4 * np.exp(-1.30 * tiempos_obs)
                 + rng.normal(0.0, 0.012, tiempos_obs.size))


def prediccion(fraccion, tasa_lenta, tasa_rapida, t=None):
    """Respuesta del modelo de dos exponenciales, adimensional."""
    t = tiempos_obs if t is None else t
    return (fraccion * np.exp(-tasa_lenta * t)
            + (1 - fraccion) * np.exp(-tasa_rapida * t))


def suma_cuadrados(fraccion, tasa_lenta, tasa_rapida):
    """Suma de cuadrados de los residuos del ajuste."""
    residuo = observaciones - prediccion(fraccion, tasa_lenta, tasa_rapida)
    return float(residuo @ residuo)


# Se fija la fraccion lenta y se optimizan las dos tasas, lo cual recorre la
# cresta del valle de ajuste en vez de mirar un solo punto.
def mejor_ajuste_con_fraccion(fraccion):
    ajuste = minimize(lambda q: suma_cuadrados(fraccion, q[0], q[1]),
                      x0=[0.45, 1.30], method="Nelder-Mead",
                      options={"xatol": 1e-10, "fatol": 1e-14, "maxiter": 20000})
    return float(ajuste.x[0]), float(ajuste.x[1]), float(ajuste.fun)


FRACCIONES = [0.45, 0.55, 0.65, 0.75, 0.85]
cresta = [(f, *mejor_ajuste_con_fraccion(f)) for f in FRACCIONES]
mejor = min(c[3] for c in cresta)
tabla = pd.DataFrame(
    [{"fraccion lenta": f, "tasa lenta (1/h)": round(k1, 4),
      "tasa rapida (1/h)": round(k2, 4), "suma de cuadrados": round(ssq, 6),
      "veces el mejor ajuste": round(ssq / mejor, 3)}
     for f, k1, k2, ssq in cresta])

figura, eje = plt.subplots()
malla_t = np.linspace(0.0, 6.5, 300)
for (f, k1, k2, ssq), color in zip(cresta, list(PALETA.values())):
    eje.plot(malla_t, prediccion(f, k1, k2, malla_t), color=color, lw=1.5,
             label=f"f = {f:.2f}, k1 = {k1:.2f} 1/h, k2 = {k2:.2f} 1/h")
eje.plot(tiempos_obs, observaciones, "o", color="black", ms=5,
         label="Observaciones")
eje.set_xlabel("Tiempo (h)")
eje.set_ylabel("Respuesta normalizada (adimensional)")
eje.legend(loc="upper right", fontsize=8.0)
eje.set_title("Equifinalidad, juegos de parametros distintos con ajuste casi igual")
plt.show()

casi_iguales = [c for c in cresta if c[3] / mejor <= 1.05]
tasas = [c[2] for c in casi_iguales]
print(f"juegos de parametros cuyo ajuste esta dentro del 5 por ciento del "
      f"mejor · {len(casi_iguales)} de {len(cresta)}")
print(f"entre ellos la tasa rapida recorre de {min(tasas):.2f} a {max(tasas):.2f} "
      f"1/h, esto es una variacion del "
      f"{100 * (max(tasas) / min(tasas) - 1):.0f} por ciento")
print("Un juego calibrado puede reproducir el pasado y fallar en el futuro sin "
      "que ninguna metrica de ajuste lo anuncie.")
tabla

## 5. Ejercicios guiados

Seis celdas incompletas con la marca `# COMPLETE:`, un valor de partida
deliberadamente incorrecto y su verificacion inmediata.

### Ejercicio 1. Problema 1-17, el numero de Biot de un cubo

El Problema 1-17 pide evaluar el numero de Biot de un cubo de 40 mm de lado y
conductividad 0.48 W/(m K) que se enfria en agua agitada con un coeficiente
convectivo de 450 W/(m2 K), y dictaminar si vale la hipotesis de solido
isotermico. La longitud caracteristica de un cuerpo compacto es el volumen sobre
el area expuesta, que para un cubo de lado a vale a sobre seis.

In [ ]:
# COMPLETE: calcule la longitud caracteristica del cubo, en metros, y su numero
# de Biot, y deje en dictamen_cubo la cadena "isotermico" o "con gradiente"
# segun el umbral de 0.1 de la Tabla 1.4.
longitud_caracteristica = 0.0   # valor de partida deliberadamente incorrecto
biot_cubo = 0.0                 # valor de partida deliberadamente incorrecto
dictamen_cubo = "isotermico"    # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("longitud caracteristica del cubo", longitud_caracteristica,
                0.0066667, 1e-4, "m")
dos = comprobar("numero de Biot del cubo", biot_cubo, 6.25, 1e-6)
if REVISAR:
    assert dictamen_cubo == "con gradiente", "con Bi = 6.25 el solido no es isotermico"
    print("[correcto ]  dictamen · con gradiente")

### Ejercicio 2. Problema 1-19, el numero de Peclet de una camara

El Problema 1-19 describe una camara de 18 m con residencia media de 45 minutos
y dispersion longitudinal de 0.9 m2/min. Segun la Tabla 1.4, el flujo se
aproxima al piston si el Peclet supera 500 y a la mezcla completa si es menor
que 5.

In [ ]:
# COMPLETE: obtenga la velocidad media como el largo sobre el tiempo de
# residencia, calcule el numero de Peclet y deje en regimen_camara una de las
# cadenas "flujo piston", "mezcla completa" o "intermedio".
peclet_camara = 0.0             # valor de partida deliberadamente incorrecto
regimen_camara = "flujo piston"  # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("numero de Peclet de la camara", peclet_camara, 8.0, 1e-6)
if REVISAR:
    assert regimen_camara == "intermedio", "el Peclet cae entre los dos umbrales"
    print("[correcto ]  regimen · intermedio")

### Ejercicio 3. Problema 1-20, el paso de tiempo por el numero de Courant

El Problema 1-20 plantea un esquema explicito para la adveccion en un rio de
velocidad 0.4 m/s con celdas de 50 m, y pide el paso de tiempo maximo para un
numero de Courant unitario.

In [ ]:
# COMPLETE: despeje el paso de tiempo maximo, en segundos, de la definicion del
# numero de Courant, que es la velocidad por el paso de tiempo sobre el tamano
# de celda.
paso_maximo = 0.0               # valor de partida deliberadamente incorrecto

In [ ]:
comprobar("paso de tiempo maximo", paso_maximo, 125.0, 1e-9, "s")

### Ejercicio 4. Cuanto dura el dia fuera del dominio de validez

Saber que el error supera el 5 por ciento por debajo de 447 W/m2 no dice todavia
cuanto pesa esa falla sobre una jornada. Traduzcala a horas. En un dia despejado
la irradiancia sigue el perfil de semiseno de 950 W/m2 entre las seis y las
dieciocho horas, de modo que hay dos tramos, uno al amanecer y otro al atardecer,
en los que la correlacion queda fuera de su banda.

In [ ]:
# COMPLETE: halle con brentq la hora de la manana, entre las 6 y las 12, en la
# que la irradiancia del perfil de semiseno de 950 W/m2 alcanza el umbral de
# 447 W/m2 que devolvio umbral_cinco. Por simetria, calcule despues las horas
# totales del dia en las que la correlacion queda fuera de su banda del 5 por
# ciento y el porcentaje que representan sobre las doce horas de sol.
hora_cruce = 0.0                # valor de partida deliberadamente incorrecto
horas_fuera = 0.0               # valor de partida deliberadamente incorrecto
porcentaje_fuera = 0.0          # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("hora del cruce por la manana", hora_cruce, 7.870816, 1e-5, "h")
dos = comprobar("horas del dia fuera de la banda", horas_fuera, 3.741632, 1e-5, "h")
tres = comprobar("porcentaje del dia fuera de la banda", porcentaje_fuera, 31.1803, 1e-5, "%")

### Ejercicio 5. Problema 1-25, propagacion de incertidumbre reutilizable

El Problema 1-25 pide escribir una rutina que devuelva la desviacion estandar de
la salida de un modelo de una entrada incierta, por primer orden y por Monte
Carlo. Las dos funciones ya estan escritas en la seccion 4, y aqui se aplican a
la pendiente longitudinal del canal, que se conoce con desviacion estandar de
0.00006 alrededor de 0.0008.

In [ ]:
# COMPLETE: defina el modelo del tirante como funcion de la pendiente, con los
# demas datos del Ejemplo 1.2 fijos, y obtenga la desviacion estandar del
# tirante por primer orden y por Monte Carlo con 2000 realizaciones.
sigma_pendiente_primer_orden = 0.0   # valor de partida deliberadamente incorrecto
sigma_pendiente_monte_carlo = 0.0    # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("desviacion por primer orden", sigma_pendiente_primer_orden,
                0.0342965, 1e-3, "m")
dos = comprobar("desviacion por Monte Carlo", sigma_pendiente_monte_carlo,
                0.0352157, 1e-2, "m")
if REVISAR and uno and dos:
    print("\nLa incertidumbre de la pendiente pesa cerca de un tercio de la que "
          "aporta la rugosidad, de modo que refinar el levantamiento "
          "topografico rinde menos que caracterizar mejor el revestimiento.")

### Ejercicio 6. Problema 1-29, el protocolo de lectura critica

El Problema 1-29 pide elegir un articulo reciente del area propia que reporte
simulaciones y aplicarle el protocolo de lectura critica de la Seccion 1.6 del
libro. Ese protocolo procede por siete preguntas, que leidas al reves describen
la estructura del informe que el estudiante debera producir.

In [ ]:
# COMPLETE: escriba las siete preguntas del protocolo de lectura critica en la
# lista PROTOCOLO, en el orden en que el libro las plantea, y responda cada una
# con "si", "no" o "no se puede saber" para el articulo que haya elegido.
PROTOCOLO = [
    "cual es la pregunta que el modelo dice responder",
]
respuestas = ["no se puede saber"]

In [ ]:
uno = comprobar("preguntas del protocolo", len(PROTOCOLO), 7, 1e-9)
dos = comprobar("respuestas registradas", len(respuestas), 7, 1e-9)
if REVISAR and uno and dos:
    print("\nLeido al reves, el protocolo describe la estructura del informe "
          "que usted debera producir en el proyecto final.")

## 6. Problemas del capitulo

- **1-21.** Demuestre que la superposicion no aplica al descenso en un acuifero
  libre, partiendo de la ecuacion de flujo en la que el espesor saturado depende
  del propio descenso. La seccion 3 de este cuaderno lo exhibe numericamente y
  falta la demostracion analitica.
- **1-27.** Disene el modelo conceptual de un sistema de aprovechamiento de agua
  lluvia para una vivienda rural de Sucre, con su frontera, sus estados,
  entradas, parametros y dominio de validez. Use como plantilla el modelo
  conceptual de la seccion 1 del cuaderno U1-02.
- **1-30.** Ensayo de dos paginas. Discuta si los asistentes de programacion
  modifican la responsabilidad de quien firma un informe de simulacion, con
  apoyo en la Tabla 1.5.

El libro cierra con una obligacion profesional concreta, que consiste en
entregar el modelo con la declaracion de lo que supone y de lo que no puede
hacer, junto con el registro de supuestos, el dominio de validez con sus limites
numericos, la incertidumbre de la respuesta y la trazabilidad entre las
versiones del codigo, de los datos y del resultado.

Sobre los asistentes de programacion la posicion del libro es explicita. El
codigo generado de forma automatica es plausible con la misma facilidad con la
que es correcto, y el ingeniero responde por el resultado que firma con
independencia de quien teclee las lineas. La practica que se promueve consiste
en declarar su uso, someter el codigo generado a las mismas pruebas que el
escrito a mano y verificar siempre el orden de magnitud esperado, la coherencia
dimensional, el caso limite y el balance que debe cerrar.

### Respuestas del estudiante a los Problemas 1-21, 1-27 y 1-30

*Escriba aqui. Para el Problema 1-21 parta de la ecuacion de Boussinesq y
muestre que el termino de transmisividad depende del descenso. Para el
Problema 1-27 entregue el diagrama de frontera y la tabla de supuestos de cuatro
columnas. Para el Problema 1-30 apoye cada afirmacion en una fila de la
Tabla 1.5.*

## Cierre

### Lista de comprobacion

Marque cada punto solo si puede hacerlo sin mirar el cuaderno.

- Declarar el dominio de validez de un modelo y cuantificar el error de extrapolarlo.
- Explicar por que un coeficiente de determinacion alto no certifica un modelo.
- Poner precio a una simplificacion mediante la comparacion entre el modelo reducido y el completo.
- Comprobar la linealidad de un problema antes de superponer soluciones elementales.
- Descomponer el error de una prediccion en sus cuatro componentes y elegir la accion que reduce cada una.
- Propagar la incertidumbre de un parametro por primer orden y contrastarla con Monte Carlo.
- Aplicar el protocolo de lectura critica a un modelo ajeno.

### Que revisar si algo no salio

- Si el ajuste de la correlacion no coincide, confirme que los diez puntos se leyeron de `datos` o se regeneraron con la semilla 20262, porque el ruido de la campana depende de ella.
- Si la serie del numero de Biot no converge, revise que las raices se busquen en intervalos que no contengan las asintotas de la tangente, esto es entre m pi y m pi mas pi medios.
- Si Monte Carlo no reproduce la desviacion publicada, compruebe que usa 5000 realizaciones, la semilla del curso y `ddof=1` en la desviacion muestral.
- Para la teoria, relea las Secciones 1.5 y 1.6 del libro, las Definiciones 1.10 y 1.11, los Ejemplos 1.4, 1.5 y 1.6, la Tabla 1.5 y los Teoremas 1.3 y 1.4.

### Declaracion del uso de asistentes de programacion

Si empleo un asistente basado en modelos de lenguaje para resolver alguna celda, declarelo en la entrega, indique en cual y describa que prueba aplico para convencerse de que el codigo es correcto. La regla de la asignatura es que el estudiante responde por el resultado que firma, con independencia de quien escriba las lineas.